# Stack time series

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import glob
import os
import shutil

import affine
import numpy as np
import pandas as pd
import rioxarray as rio  # noqa: F401
import xarray as xr
from pyproj import CRS

In [ ]:
import evaspa.timeseries.io_handler as ioh
import evaspa.timeseries.stack_handler as sth
import evaspa.timeseries.status_handler as sh
from evaspa.common.io import write_dataset
from evaspa.timeseries.constant import TimeSeriesVar as TSVar

### Create time series

In [ ]:
# Setup data
window_size = 7  # Time
x_size = 1
y_size = 2
today = dt.datetime.now(tz=dt.timezone.utc).date()
dates = np.array(pd.date_range(end=today, periods=window_size).to_list())
# Define transform: (origin_x, origin_y), pixel size = 0.1 degree
transform = affine.Affine.translation(0, 40) * affine.Affine.scale(0.1, -0.1)
# Generate x/y coordinates from transform
x_coords = [(transform * (i, 0))[0] for i in range(x_size)]  # x: lon
y_coords = [(transform * (0, j))[1] for j in range(y_size)]  # y: lat
# Create ET time series data
et_ts = xr.Dataset(
    {
        TSVar.ET.value: (
            ["time", "y", "x"],
            np.transpose(
                np.array(
                    [
                        [
                            [1.2, 1.4, 1.6, 1.8, 1.8, 1.8, np.nan],
                            [1.4, 1.4, 1.4, 1.4, 1.4, 1.4, np.nan],
                        ]
                    ]
                ),
                (2, 1, 0),
            ),
        ),
        TSVar.FLAGS.value: (
            ["time", "y", "x"],
            np.transpose(
                np.array(
                    [
                        [
                            [0, 386, 386, 0, 131, 275, 1],
                            [132, 0, 131, 259, 387, 531, 1],
                        ]
                    ],
                    dtype=sh.STATUS_TYPE,
                ),
                (2, 1, 0),
            ),
        ),
    },
    coords={
        "time": dates,
        "x": x_coords,
        "y": y_coords,
    },
)
# Add CRS and transform metadata (compatible with rioxarray)
et_ts = et_ts.rio.write_crs(CRS(4236))
et_ts = et_ts.rio.write_transform(transform)
# Create radiation time series
radiation_ts = xr.Dataset(
    {
        TSVar.RADIATION.value: (
            ["time", "y", "x"],
            np.transpose(
                np.array(
                    [
                        [
                            [3.5e7, 3.5e7, 3.5e7, 3.5e7, 3.5e7, 3.5e7, 3.5e7],
                            [3.5e7, 3.5e7, 3.5e7, 3.5e7, 3.5e7, 3.5e7, 3.5e7],
                        ]
                    ]
                ),
                (2, 1, 0),
            ),
        ),
    },
    coords={
        "time": dates,
        "x": x_coords,
        "y": y_coords,
    },
)
# Add CRS and transform metadata (compatible with rioxarray)
radiation_ts = radiation_ts.rio.write_crs(CRS(4236))
radiation_ts = radiation_ts.rio.write_transform(transform)
# Create ET single dates
acquisition_dates = [1, 3, 5, 6]
et_sd = xr.Dataset(
    {
        TSVar.ET.value: (
            ["time", "y", "x"],
            np.transpose(
                np.array(
                    [[[np.nan, 1.8, np.nan, 2.4], [1.4, np.nan, 2.2, np.nan]]]
                ),
                (2, 1, 0),
            ),
        ),
        TSVar.FLAGS.value: (
            ["time", "y", "x"],
            np.transpose(np.array([[[0, 0, 0, 0], [0, 1, 0, 1]]]), (2, 1, 0)),
        ),
    },
    coords={
        "time": dates[acquisition_dates],
        "x": x_coords,
        "y": y_coords,
    },
)
# Add CRS and transform metadata (compatible with rioxarray)
et_sd = et_sd.rio.write_crs(CRS(4236))
et_sd = et_sd.rio.write_transform(transform)
# Create a DEM dataset
dem = xr.Dataset(
    {
        TSVar.HEIGHT.value: (
            ["y", "x"],
            np.array([[10.0], [10.0]]),
        ),
        TSVar.SLOPE.value: (
            ["y", "x"],
            np.array([[5.0], [5.0]]),
        ),
        TSVar.ASPECT.value: (
            ["y", "x"],
            np.array([[10.0], [10.0]]),
        ),
    },
    coords={
        "x": x_coords,
        "y": y_coords,
    },
)
# Add CRS and transform metadata (compatible with rioxarray)
dem = dem.rio.write_crs(CRS(4236))
dem = dem.rio.write_transform(transform)

### Write time series 

In [ ]:
output_dir = os.path.join(os.getcwd(), "out")
if os.path.isdir(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

In [ ]:
ioh.write_timeseries(
    et_ts,
    root_name="et_time_series",
    directory=output_dir,
)
ioh.write_timeseries(radiation_ts, root_name="radiation", directory=output_dir)
ioh.write_timeseries(et_sd, root_name="et_single_date", directory=output_dir)
write_dataset(dem, filename="dem.tif", directory=output_dir)

### Input data

In [ ]:
# Generate lists of file paths
et_ts_files = [
    os.path.join(output_dir, file)
    for file in glob.glob("et_time_series_*.tif", root_dir=output_dir)
]
radiation_files = [
    os.path.join(output_dir, file)
    for file in glob.glob("radiation_*.tif", root_dir=output_dir)
]
et_files = [
    os.path.join(output_dir, file)
    for file in glob.glob("et_single_date_*.tif", root_dir=output_dir)
]
dem_file = os.path.join(output_dir, "dem.tif")

In [ ]:
print("ET time series = ", et_ts_files)
print("Radiation = ", radiation_files)
print("ET single date = ", et_files)
print("DEM = ", dem_file)

In [ ]:
# Create radiation time series with missing files
radiation_dates = [0, 1, 2, 3, 5]
radiation_files_with_missing_dates = [
    radiation_files[i] for i in radiation_dates
]
print(radiation_files_with_missing_dates)

### Read and stack (no missing radiation data)

In [ ]:
config = {
    "et_time_series": et_ts_files,
    "radiation": radiation_files,
    "et_single_date": et_files,
    "dem": None,
}

In [ ]:
et_ts, radiation_ts, et_sd, dem = ioh.read_input(config)

In [ ]:
sth.stack_time_series(et_ts, radiation_ts, dem)

### Read and stack (missing radiation data)

In [ ]:
config = {
    "et_time_series": et_ts_files,
    "radiation": radiation_files_with_missing_dates,
    "et_single_date": et_files,
    "dem": None,
}

In [ ]:
et_ts, radiation_ts, et_sd, dem = ioh.read_input(config)

In [ ]:
sth.stack_time_series(et_ts, radiation_ts, dem)

### Filter and stack data

In [ ]:
input_config = {
    "et_time_series": et_ts_files,
    "radiation": radiation_files,
    "et_single_date": et_files,
    "dem": None,
}

In [ ]:
et_ts, radiation_ts, et_sd, dem = ioh.read_input(input_config)

In [ ]:
stack_config = {
    "et_single_date_filtering": {
        "flags": {"op": "==", "value": 0},
    }
}

In [ ]:
stack_config

In [ ]:
config = sth.TimeSeriesStackConfig.model_validate(stack_config)

In [ ]:
ts, feed = sth.run(
    et_ts,
    radiation_ts,
    et_sd,
    dem,
    et_single_date_filtering=config.et_single_date_filtering.model_dump(),
)

In [ ]:
ts

In [ ]:
feed

In [ ]:
input_config = {
    "et_time_series": et_ts_files,
    "et_single_date": et_files,
}

In [ ]:
et_ts, radiation_ts, et_sd, dem = ioh.read_input(input_config)

In [ ]:
ts, feed = sth.run(
    et_ts,
    radiation_ts,
    et_sd,
    dem,
    et_single_date_filtering=config.et_single_date_filtering.model_dump(),
)

In [ ]:
ts

In [ ]:
feed